# Data Collection

In [13]:
import pandas as pd
import requests
import os
import time

Load the location data

In [14]:
location_data = pd.read_csv(
    "../Data/Reference/location_master.csv"
)

location_data.head()

,district,latitude,longitude,elevation
0,Colombo,6.93548,79.84868,7.0
1,Gampaha,7.08970,79.99250,21.0
2,Kalutara,6.58310,79.95930,7.0
3,Kandy,7.29060,80.63360,500.0
4,Matale,7.46980,80.62170,366.0


In [15]:
print("Locations:", len(location_data))

Locations: 25


In [16]:
os.makedirs(
    "../Data/Raw/Open_Meteo",
    exist_ok=True
)
os.makedirs(
    "../Data/Raw/PVGIS",
    exist_ok=True
)

Test Open-Meteo historical weather data

In [17]:
colombo = location_data[
    location_data["district"] == "Colombo"
].iloc[0]

latitude = colombo["latitude"]
longitude = colombo["longitude"]

weather_url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

weather_params = {
    "start_date": "2022-01-01",
    "end_date": "2023-12-31",
    "hourly": [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "cloud_cover",
    "wind_speed_10m",
    "shortwave_radiation",
    "direct_radiation",
    "diffuse_radiation",
    "direct_normal_irradiance",
    "global_tilted_irradiance",
    "sunshine_duration",
    "is_day"
],
    "tilt": 10,
    "azimuth": 0,
    "timezone": "GMT",
    "models": "gfs_seamless"
}

test_params = weather_params.copy()

test_params["latitude"] = latitude
test_params["longitude"] = longitude
test_params["start_date"] = "2022-01-01"
test_params["end_date"] = "2022-01-07"

response = requests.get(
    weather_url,
    params=test_params
)

print(response.status_code)

200


In [18]:
weather_data = response.json()

print("Timezone:", weather_data.get("timezone"))
print("UTC offset:", weather_data.get("utc_offset_seconds"))

print("\nHourly units:")
print(weather_data.get("hourly_units"))

print("\nHourly variables:")
print(weather_data.get("hourly").keys())

Timezone: GMT
UTC offset: 0

Hourly units:
{'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'precipitation': 'mm', 'cloud_cover': '%', 'wind_speed_10m': 'km/h', 'shortwave_radiation': 'W/m²', 'direct_radiation': 'W/m²', 'diffuse_radiation': 'W/m²', 'direct_normal_irradiance': 'W/m²', 'global_tilted_irradiance': 'W/m²', 'sunshine_duration': 's', 'is_day': ''}

Hourly variables:
dict_keys(['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'cloud_cover', 'wind_speed_10m', 'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'direct_normal_irradiance', 'global_tilted_irradiance', 'sunshine_duration', 'is_day'])


Test PVGIS solar PV data



In [19]:
pvgis_url = "https://re.jrc.ec.europa.eu/api/v5_3/seriescalc"

pvgis_params = {
    "lat": latitude,
    "lon": longitude,
    "startyear": 2022,
    "endyear": 2023,
    "pvcalculation": 1,
    "peakpower": 1,
    "pvtechchoice": "crystSi",
    "mountingplace": "free",
    "loss": 14,
    "angle": 10,
    "aspect": 0,
    "usehorizon": 0,
    "outputformat": "json"
}

pvgis_response = requests.get(
    pvgis_url,
    params=pvgis_params
)

print(pvgis_response.status_code)

200


 Collect historical weather data



In [20]:
for _, row in location_data.iterrows():

    file_name = (
        row["district"]
        .lower()
        .replace(" ", "_")
        + ".csv"
    )

    file_path = (
        "../Data/Raw/Open_Meteo/"
        + file_name
    )

    if os.path.exists(file_path):
        print(row["district"], "already saved")
        continue

    params = weather_params.copy()
    params["latitude"] = row["latitude"]
    params["longitude"] = row["longitude"]

    response = requests.get(
        weather_url,
        params=params
    )

    if response.status_code == 429:
        time.sleep(10)

        response = requests.get(
            weather_url,
            params=params
        )

    if response.status_code == 200:

        df = pd.DataFrame(
            response.json()["hourly"]
        )

        df["district"] = row["district"]
        df["latitude"] = row["latitude"]
        df["longitude"] = row["longitude"]
        df["elevation"] = row["elevation"]

        df.to_csv(
            file_path,
            index=False
        )

        print(row["district"], len(df))

    else:
        print(
            row["district"],
            "failed",
            response.status_code
        )

    time.sleep(2)

Colombo already saved
Gampaha already saved
Kalutara already saved
Kandy already saved
Matale already saved
Nuwara Eliya already saved
Galle already saved
Matara already saved
Hambantota already saved
Jaffna already saved
Kilinochchi already saved
Mannar already saved
Mullaitivu already saved
Vavuniya already saved
Batticaloa already saved
Ampara already saved
Trincomalee already saved
Kurunegala already saved
Puttalam already saved
Anuradhapura already saved
Polonnaruwa already saved
Badulla already saved
Monaragala already saved
Ratnapura already saved
Kegalle already saved


Validate the collected weather data



In [21]:
saved_weather = [
    f for f in os.listdir("../Data/Raw/Open_Meteo")
    if f.endswith(".csv")
]

row_counts = []
missing_values = 0

for file in saved_weather:

    df = pd.read_csv(
        "../Data/Raw/Open_Meteo/" + file
    )

    row_counts.append(len(df))
    missing_values += df.isnull().sum().sum()

print("Saved weather files:", len(saved_weather))
print("Total rows:", sum(row_counts))
print(
    "Different row counts:",
    sorted(set(row_counts))
)
print("Missing values:", missing_values)

Saved weather files: 25
Total rows: 438000
Different row counts: [17520]
Missing values: 8625


Collect PVGIS solar PV data



In [22]:
for _, row in location_data.iterrows():

    file_name = (
        row["district"]
        .lower()
        .replace(" ", "_")
        + ".csv"
    )

    file_path = (
        "../Data/Raw/PVGIS/"
        + file_name
    )

    if os.path.exists(file_path):
        print(row["district"], "already saved")
        continue

    params = pvgis_params.copy()
    params["lat"] = row["latitude"]
    params["lon"] = row["longitude"]

    response = requests.get(
        pvgis_url,
        params=params
    )

    if response.status_code == 200:

        data = response.json()

        df = pd.DataFrame(
            data["outputs"]["hourly"]
        )

        df["district"] = row["district"]
        df["latitude"] = row["latitude"]
        df["longitude"] = row["longitude"]

        df.to_csv(
            file_path,
            index=False
        )

        print(row["district"], len(df))

    else:
        print(
            row["district"],
            "failed",
            response.status_code
        )

    time.sleep(1)

Colombo already saved
Gampaha already saved
Kalutara already saved
Kandy already saved
Matale already saved
Nuwara Eliya already saved
Galle already saved
Matara already saved
Hambantota already saved
Jaffna already saved
Kilinochchi already saved
Mannar already saved
Mullaitivu already saved
Vavuniya already saved
Batticaloa already saved
Ampara already saved
Trincomalee already saved
Kurunegala already saved
Puttalam already saved
Anuradhapura already saved
Polonnaruwa already saved
Badulla already saved
Monaragala already saved
Ratnapura already saved
Kegalle already saved


Validate the collected PVGIS data

In [23]:
saved_pvgis = [
    f for f in os.listdir("../Data/Raw/PVGIS")
    if f.endswith(".csv")
]

row_counts = []
missing_values = 0

for file in saved_pvgis:

    df = pd.read_csv(
        "../Data/Raw/PVGIS/" + file
    )

    row_counts.append(len(df))
    missing_values += df.isnull().sum().sum()

print("Saved PVGIS files:", len(saved_pvgis))
print("Total rows:", sum(row_counts))
print(
    "Different row counts:",
    sorted(set(row_counts))
)
print("Missing values:", missing_values)

Saved PVGIS files: 25
Total rows: 438000
Different row counts: [17520]
Missing values: 0
